# Fairness processing methods

## Requirements

In [6]:
import numpy as np
import pandas as pd 

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split 

from aif360.datasets import GermanDataset
from aif360.sklearn.metrics import statistical_parity_difference
from aif360.sklearn.preprocessing import Reweighing, ReweighingMeta
from aif360.sklearn.inprocessing import AdversarialDebiasing

import tensorflow as tf

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

## Initial elements

### Data

In [7]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [8]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0



### Response and predictors


In [9]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [10]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute


### Outer evaluation method: simple-validation (train-test split)

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

## Pre-processing

In [12]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)
log_reg_estimator.fit(X_train, Y_train)
Y_test_hat = log_reg_estimator.predict(X_test)
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set

In [13]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.32539682539682535

In [14]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6396825396825396

### Reweighing

In [15]:
class ReweighingMetaEstimator(BaseEstimator, ClassifierMixin):
   
    def __init__(self, estimator, prot_attr):
        
        self.estimator = estimator
        self.prot_attr = prot_attr

    def fit(self, X, y):

        # X must be a Pandas DataFrame
        
        A = X[self.prot_attr]
        reweigher_ = Reweighing(prot_attr=A)
        self.meta_estimator = ReweighingMeta(estimator=self.estimator, reweigher=reweigher_)
        self.meta_estimator.fit(X, y)
        self.classes_ = self.meta_estimator.classes_

        return self
    
    def predict(self, X):

        return self.meta_estimator.predict(X)
    
    def predict_proba(self, X):

        return self.meta_estimator.predict_proba(X)

In [16]:
reweighing_log_reg_estimator = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)
reweighing_log_reg_estimator.fit(X=X_train, y=Y_train)
Y_test_hat_reweighing = reweighing_log_reg_estimator.predict(X_test)

In [17]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat_reweighing, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.17658730158730163

In [18]:
balanced_accuracy_score(y_pred=Y_test_hat_reweighing, y_true=Y_test)

0.6396825396825396

## In-processing

In [19]:
A_train = X_train[sens_variable_name] # sensitive variable / protected attribute in train set

### Adversarial Debiasing

In [31]:
class AdversarialDebiasingEstimator(BaseEstimator, ClassifierMixin):
   
    def __init__(self, prot_attr, scope_name, adversary_loss_weight, num_epochs, 
                 batch_size, classifier_num_hidden_units, debias, verbose, random_state):
        
        self.prot_attr = prot_attr
        self.scope_name = scope_name
        self.adversary_loss_weight  = adversary_loss_weight
        self.num_epochs  = num_epochs
        self.batch_size  = batch_size
        self.classifier_num_hidden_units  = classifier_num_hidden_units
        self.debias  = debias
        self.verbose  = verbose
        self.random_state  = random_state

    def fit(self, X, y):

        # X must be a Pandas DataFrame

        A = X[self.prot_attr]
        self.estimator = AdversarialDebiasing(prot_attr=A, # Protected attribute
                                              scope_name=self.scope_name, # Scope for TensorFlow variables    
                                              adversary_loss_weight=self.adversary_loss_weight, # Weight of adversarial loss
                                              num_epochs=self.num_epochs, # Number of training epochs
                                              batch_size=self.batch_size, # Batch size for training
                                              classifier_num_hidden_units=self.classifier_num_hidden_units, # Number of hidden units in the classifier
                                              debias=self.debias, # Apply debiasing
                                              verbose=self.verbose, # Print progress
                                              random_state=self.random_state # Ensure reproducibility
                                            )

        self.estimator.fit(X, y)

        return self
    
    def predict(self, X):

        return self.estimator.predict(X)
    
    def predict_proba(self, X):

        return self.estimator.predict_proba(X)

In [32]:
# Set up TensorFlow session (required by AdversarialDebiasing)
tf_session = tf.compat.v1.Session
# disable_eager_execution is required as well by TensorFlow
tf.compat.v1.disable_eager_execution()

In [33]:
adversarial_debiasing = AdversarialDebiasingEstimator(    
                                                        prot_attr=sens_variable_name,               
                                                        scope_name='classifier',            
                                                        adversary_loss_weight=0.1,           
                                                        num_epochs=50,                      
                                                        batch_size=128,                     
                                                        classifier_num_hidden_units=200,     
                                                        debias=True,                        
                                                        verbose=True,                     
                                                        random_state=123                   
                                                        )

In [37]:
adversarial_debiasing.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

AdversarialDebiasingEstimator(adversary_loss_weight=0.1, batch_size=128,
                              classifier_num_hidden_units=200, debias=True,
                              num_epochs=50, prot_attr='age', random_state=123,
                              scope_name='classifier', verbose=True)

In [39]:
Y_test_hat = adversarial_debiasing.predict(X_test)
Y_test_hat

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [40]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [41]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908